# Bays & Brady (2024) Figure 5 — GP-Based Equivalent

**Scientific question:** Does the GP population coding model reproduce the key phenomenology from "Changing Concepts in Working Memory"?

Three panels:
- **(a)** Pooled error distribution across all set sizes — heavy tails
- **(b)** Actual SD (degrees) vs set size 1–8 — continuous rise, no plateau
- **(c)** Model SD vs theoretical $\sqrt{l}$ scaling from DN activity cap

The DN activity cap predicts: per-item gain $= \gamma N / l$, so Fisher info $\propto 1/l$, giving SD $\propto \sqrt{l}$. Deviations from $\sqrt{l}$ reveal effects of GP heterogeneity beyond simple gain-scaling.

In [ ]:
# ============================================================
# PATH SETUP + CORE IMPORTS
# ============================================================
import sys
from pathlib import Path

# Notebook lives in experiments/nature/
PROJECT_ROOT = str(Path.cwd().parents[1])
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time

from core.encoder.gaussian_process import periodic_rbf_kernel, sample_gp_function
from core.encoder.poisson_spike import generate_spikes
from core.encoder.divisive_normalization import dn_pointwise

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# ============================================================
# Single-location full Poisson ML decoder + circular stats
# ============================================================

def compute_log_likelihood(counts, g, T_d):
    log_g = np.log(np.maximum(g, 1e-30))
    return counts @ log_g - T_d * np.sum(g, axis=0)

def compute_circular_error(theta_true, theta_hat):
    return np.angle(np.exp(1j * (theta_hat - theta_true)))

def circular_variance(errors):
    m1 = np.mean(np.exp(1j * errors))
    rho1 = np.clip(np.abs(m1), 1e-15, 1.0 - 1e-10)
    return -2.0 * np.log(rho1)

def circular_sd_degrees(errors):
    return np.degrees(np.sqrt(circular_variance(errors)))

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
M = 100; N_THETA = 64; N_TRIALS = 5000; T_D = 0.1; SIGMA_SQ = 1e-6
LAMBDA_BASE = 0.5; GAMMA = 100.0
SET_SIZES = [1, 2, 3, 4, 5, 6, 7, 8]
SEED = 42; N_SEEDS = 3; N_BINS = 60

## Population generation + trial engine

Single-location pipeline with the $\gamma/N$ shortcut: at set size $N$, the effective gain is $\gamma/N$, simulating DN's resource division without multi-location encoding.

In [ ]:
def generate_population(M, n_theta, lengthscale, seed):
    rng = np.random.RandomState(seed)
    thetas = np.linspace(-np.pi, np.pi, n_theta, endpoint=False)
    K = periodic_rbf_kernel(thetas, lengthscale)
    g = np.zeros((M, n_theta))
    for i in range(M):
        g[i] = np.exp(sample_gp_function(K, rng))
    return thetas, g

def run_trials(g, thetas, gamma, T_d, sigma_sq, n_trials, rng):
    M, n_theta = g.shape
    errors = np.empty(n_trials)
    for t in range(n_trials):
        idx_true = rng.randint(n_theta)
        rates = dn_pointwise(g[:, idx_true], gamma, sigma_sq)
        counts = generate_spikes(rates, T_d, rng)
        ll = compute_log_likelihood(counts, g, T_d)
        errors[t] = compute_circular_error(thetas[idx_true], thetas[np.argmax(ll)])
    return errors

## Sweep set sizes 1–8

In [ ]:
t0 = time.time()
all_seeds = []

for s in range(N_SEEDS):
    cseed = SEED + s * 1000
    thetas, g = generate_population(M, N_THETA, LAMBDA_BASE, cseed)
    seed_data = {}
    for N in SET_SIZES:
        effective_gamma = GAMMA / N
        rng = np.random.RandomState(cseed + N)
        errors = run_trials(g, thetas, effective_gamma, T_D, SIGMA_SQ, N_TRIALS, rng)
        seed_data[N] = {'errors': errors, 'sd_deg': circular_sd_degrees(errors)}
        print(f"  seed={s} N={N}: SD={seed_data[N]['sd_deg']:.2f} deg")
    all_seeds.append(seed_data)

# Aggregate
summary = {}
for N in SET_SIZES:
    sds = [sd[N]['sd_deg'] for sd in all_seeds]
    summary[N] = {
        'sd_mean': np.mean(sds),
        'sd_se': np.std(sds, ddof=1) / np.sqrt(N_SEEDS) if N_SEEDS > 1 else 0,
    }

# Pooled errors for panel (a)
pooled = np.concatenate([sd[N]['errors'] for sd in all_seeds for N in SET_SIZES])
bin_edges = np.linspace(-np.pi, np.pi, N_BINS + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
counts_hist, _ = np.histogram(pooled, bins=bin_edges)
bw = bin_edges[1] - bin_edges[0]
pooled_density = counts_hist / (len(pooled) * bw)
pooled_sd = circular_sd_degrees(pooled)

# Theoretical sqrt(l)
sd_at_1 = summary[1]['sd_mean']
theoretical_sd = sd_at_1 * np.sqrt(np.array(SET_SIZES, dtype=float))

print(f"\nDone in {time.time()-t0:.1f}s, pooled SD = {pooled_sd:.1f} deg")

## Three-panel figure

In [ ]:
ns = np.array(SET_SIZES, dtype=float)
sd_mean = np.array([summary[N]['sd_mean'] for N in SET_SIZES])
sd_se = np.array([summary[N]['sd_se'] for N in SET_SIZES])

RED, BLACK, BLUE, GRAY = '#CC2222', '#222222', '#2255AA', '#AAAAAA'

fig = plt.figure(figsize=(14.5, 4.5))
gs = gridspec.GridSpec(1, 3, width_ratios=[1.2, 1, 1], wspace=0.35)

# Panel (a): pooled error distribution
ax_a = fig.add_subplot(gs[0])
ax_a.bar(np.degrees(bin_centers), pooled_density * np.pi/180,
         width=np.degrees(bw)*0.85, color=GRAY, edgecolor='#888', lw=0.3, alpha=0.7)
ax_a.annotate(f'Actual SD = {pooled_sd:.1f}\u00b0', xy=(0.97, 0.95),
              xycoords='axes fraction', fontsize=9, ha='right', va='top',
              bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.9))
ax_a.set_xlim(-180, 180); ax_a.set_xticks([-180,-90,0,90,180])
ax_a.set_xlabel('Estimation error (degrees)'); ax_a.set_ylabel('Probability density')
ax_a.text(-0.14, 1.06, r'$\mathbf{a}$', transform=ax_a.transAxes, fontsize=15, fontweight='bold', va='top')

# Panel (b): actual SD vs set size
ax_b = fig.add_subplot(gs[1])
ax_b.errorbar(ns, sd_mean, yerr=sd_se, fmt='o-', color=BLACK, lw=1.5, ms=5, capsize=3, label='Actual SD')
ax_b.set_xlim(0.5, 8.5); ax_b.set_xticks(SET_SIZES); ax_b.set_ylim(bottom=0)
ax_b.set_xlabel('Set size'); ax_b.set_ylabel('SD (degrees)')
ax_b.legend(fontsize=8); ax_b.set_title('GP Model')
ax_b.text(-0.14, 1.06, r'$\mathbf{b}$', transform=ax_b.transAxes, fontsize=15, fontweight='bold', va='top')

# Panel (c): model vs theoretical sqrt(l)
ax_c = fig.add_subplot(gs[2])
ax_c.plot(ns, sd_mean, 'o-', color=RED, lw=1.5, ms=5, label='GP model')
ax_c.fill_between(ns, sd_mean-sd_se, sd_mean+sd_se, color=RED, alpha=0.15)
ns_smooth = np.linspace(1, 8, 100)
ax_c.plot(ns_smooth, sd_at_1*np.sqrt(ns_smooth), '-', color=BLUE, lw=1.5, alpha=0.7,
          label=r'Theory: SD $\propto \sqrt{l}$')
ax_c.set_xlim(0.5, 8.5); ax_c.set_xticks(SET_SIZES); ax_c.set_ylim(bottom=0)
ax_c.set_xlabel('Set size'); ax_c.set_ylabel('SD (degrees)')
ax_c.legend(fontsize=8); ax_c.set_title('Model vs Theory')
ax_c.text(-0.14, 1.06, r'$\mathbf{c}$', transform=ax_c.transAxes, fontsize=15, fontweight='bold', va='top')

fig.suptitle(f'Bays & Brady (2024) Fig 5 — GP Model ($\lambda$={LAMBDA_BASE}, $\gamma$={GAMMA} Hz, M={M})',
             fontsize=12, fontweight='bold', y=0.97)
plt.tight_layout(); plt.show()